# Text-to-SQL: LLM-Powered Database Querying

Companion notebook for the [Text-to-SQL wiki page](https://ml-viz-ruby.vercel.app/wiki/text-to-sql).

We implement schema-aware prompt construction, basic BM25-based schema linking, and the execution-validate-correct loop using SQLite.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import re, sqlite3, tempfile, os
import numpy as np

## 1 — Schema-aware prompt construction

In [ ]:
SCHEMA = {
    "orders": {
        "columns": ["order_id INT", "customer_id INT", "amount DECIMAL(10,2)", 
                    "status VARCHAR(20)", "created_at TIMESTAMP"],
        "description": "Customer orders with amount and status"
    },
    "customers": {
        "columns": ["customer_id INT", "name VARCHAR(100)", "state VARCHAR(2)", "email VARCHAR(200)"],
        "description": "Customer profile information"
    },
    "products": {
        "columns": ["product_id INT", "name VARCHAR(200)", "category VARCHAR(50)", "price DECIMAL(10,2)"],
        "description": "Product catalog"
    }
}

def schema_to_create_statements(schema):
    """Convert schema dict to CREATE TABLE statements for the prompt."""
    stmts = []
    for table, info in schema.items():
        cols = ',\n  '.join(info['columns'])
        stmts.append(f"CREATE TABLE {table} (\n  {cols}\n);")
    return '\n\n'.join(stmts)

def build_text2sql_prompt(question, schema, dialect="SQLite"):
    return f"""You are an expert SQL query writer for {dialect}.
Given the database schema below, write a SQL query to answer the question.
Return ONLY the SQL query, no explanation.

Schema:
{schema_to_create_statements(schema)}

Question: {question}

SQL:"""

prompt = build_text2sql_prompt(
    "How many orders were placed by customers in California?",
    SCHEMA
)
print(prompt)

## 2 — Schema linking: find relevant tables

In [ ]:
def schema_link(question, schema):
    """Simple keyword-based schema linking: return tables relevant to the question."""
    q_tokens = set(re.findall(r'[a-zA-Z]+', question.lower()))
    relevant = []
    for table, info in schema.items():
        table_tokens = set(re.findall(r'[a-zA-Z]+', (table + ' ' + info['description']).lower()))
        col_tokens = set(re.findall(r'[a-zA-Z]+', ' '.join(info['columns']).lower()))
        all_tokens = table_tokens | col_tokens
        overlap = len(q_tokens & all_tokens)
        if overlap > 0:
            relevant.append((table, overlap))
    relevant.sort(key=lambda x: -x[1])
    return [t for t, _ in relevant]

questions = [
    "How many orders were placed last month?",
    "What is the average product price by category?",
    "Find customers in New York who placed orders over $100"
]
for q in questions:
    tables = schema_link(q, SCHEMA)
    print(f"Q: {q[:60]}")
    print(f"   Relevant tables: {tables}\n")

## 3 — Execute-validate-correct loop with SQLite

In [ ]:
# Create an in-memory SQLite database for testing
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY, customer_id INTEGER,
    amount REAL, status TEXT, created_at TEXT
);
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, name TEXT, state TEXT, email TEXT
);
INSERT INTO customers VALUES (1,'Alice','CA','a@x.com'),(2,'Bob','NY','b@x.com'),(3,'Carol','CA','c@x.com');
INSERT INTO orders VALUES (1,1,150.0,'completed','2024-01-15'),
                          (2,2,80.0,'completed','2024-01-16'),
                          (3,3,200.0,'pending','2024-01-17'),
                          (4,1,50.0,'completed','2024-01-18');
""")

def execute_and_correct(sql_candidates, conn, max_retries=3):
    """Try each SQL candidate; return first one that executes successfully."""
    for i, sql in enumerate(sql_candidates):
        try:
            cur = conn.execute(sql)
            result = cur.fetchall()
            return sql, result, i
        except Exception as e:
            print(f"  Attempt {i+1} failed: {e}")
    return None, None, -1

# Simulate LLM generating SQL (first attempt has a bug, second is correct)
candidates = [
    "SELECT COUNT(*) FROM order WHERE state = 'CA'",   # wrong table name
    "SELECT COUNT(*) FROM orders o JOIN customers c ON o.customer_id = c.customer_id WHERE c.state = 'CA'",
]

sql, result, attempt = execute_and_correct(candidates, conn)
print(f"\nSucceeded on attempt {attempt+1}: {sql}")
print(f"Result: {result}")

## ✏️ Your turn

In [ ]:
def count_tokens_in_prompt(prompt_text):
    """Estimate token count (rough: split on whitespace and punctuation)."""
    # TODO(you): count the approximate number of tokens.
    # Simple heuristic: 1 token ≈ 4 characters (GPT-4 BPE rule of thumb)
    return ...

full_prompt = build_text2sql_prompt("How many orders per customer?", SCHEMA)
tokens = count_tokens_in_prompt(full_prompt)
print(f"Estimated tokens in prompt: {tokens}")
print(f"At $0.01/1K tokens (GPT-4o), cost: ${tokens/1000 * 0.01:.5f}")

<details><summary>Solution</summary>

```python
def count_tokens_in_prompt(prompt_text):
    return len(prompt_text) // 4
```

This is a rough BPE estimate. For exact token counting use `tiktoken` (OpenAI's tokenizer): `import tiktoken; enc = tiktoken.get_encoding("cl100k_base"); len(enc.encode(text))`.
</details>